# PUMA V13.2 — 00 Preprocess

Validate the dataset, build the NPY artifacts, and assign five stratified Stage-1 folds.
Run once, unless the raw dataset or the preprocessing configuration changes.

Fold assignment is capacity-constrained: every fold receives an equal ROI quota, class
balance is optimised inside that quota, and the last cell rejects a degenerate split
before any training starts.


In [ ]:
# Project bootstrap. Runs on the "SymbioPan (uv .venv)" kernel on this workstation, and
# on Colab without changes. No %pip here: dependencies come from setup_local.sh (uv) or,
# on Colab, from `!pip install -q -r requirements_colab.txt` in a scratch cell.
from pathlib import Path
import os
import sys

try:
    import google.colab  # type: ignore
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/Research/PUMA')
    ON_COLAB = True
except ImportError:
    PROJECT_DIR = Path.cwd().resolve()
    ON_COLAB = False

PROJECT_DIR = PROJECT_DIR.expanduser().resolve()
if not (PROJECT_DIR / 'puma').is_dir():
    raise RuntimeError(
        f"{PROJECT_DIR} is not the project root (no puma/ package here). "
        "Start JupyterLab from the project root, or set PROJECT_DIR explicitly."
    )
os.chdir(PROJECT_DIR)

# Drop any stale puma modules so an edited package is always re-imported.
for module_name in [m for m in sys.modules if m == 'puma' or m.startswith('puma.')]:
    del sys.modules[module_name]
if str(PROJECT_DIR) in sys.path:
    sys.path.remove(str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR))

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('PUMA_STAGE2_CROP_CACHE_MB', '512')
os.environ.setdefault('PUMA_V132_AUTO_OOM_FALLBACK', '1')

# GPU selection. On a workstation with two or more GPUs this pins training to GPU 1,
# leaving GPU 0 for the display and for other jobs; with a single GPU (or on Colab) it
# falls back to GPU 0. It must happen before torch is imported, because
# CUDA_VISIBLE_DEVICES is only read when the CUDA driver initialises -- puma.gpu imports
# no torch for that reason. An existing CUDA_VISIBLE_DEVICES is respected, so
#     CUDA_VISIBLE_DEVICES=0 ./.venv/bin/jupyter lab
# still overrides this. The selected GPU becomes cuda:0 inside torch.
from puma.gpu import describe_selection, select_cuda_device

PREFERRED_GPU_INDEX = 1
gpu_selection = select_cuda_device(PREFERRED_GPU_INDEX)

print('PROJECT_DIR =', PROJECT_DIR)
print('python      =', sys.executable)
print()
print(describe_selection(gpu_selection))


In [ ]:
# Verify the kernel is the uv venv and every dependency imports.
# Dependencies are managed by uv, not by %pip:
#     uv pip install -r requirements_colab.txt      (inside .venv)
import importlib
import sys
from pathlib import Path

if not ON_COLAB:
    expected = (PROJECT_DIR / '.venv' / 'bin' / 'python').resolve()
    if Path(sys.executable).resolve() != expected:
        raise RuntimeError(
            f"Wrong kernel: {sys.executable}\nExpected: {expected}\n"
            "In JupyterLab pick Kernel > Change Kernel > 'SymbioPan (uv .venv)'."
        )

for name in ('numpy', 'pandas', 'scipy', 'tifffile', 'shapely', 'rasterio',
             'torch', 'timm', 'huggingface_hub', 'safetensors', 'tqdm', 'psutil'):
    module = importlib.import_module(name)
    print(f"{name:16s} {getattr(module, '__version__', 'unknown')}")

import torch

print('\ncuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    if torch.cuda.device_count() != 1:
        raise RuntimeError(
            f"Expected exactly one visible GPU after selection, saw "
            f"{torch.cuda.device_count()}. CUDA_VISIBLE_DEVICES="
            f"{os.environ.get('CUDA_VISIBLE_DEVICES')!r}. Restart the kernel and Run All."
        )
    properties = torch.cuda.get_device_properties(0)
    physical = gpu_selection.get('selected_index')
    print(f"cuda:0 = physical GPU {physical}: {properties.name}  "
          f"{properties.total_memory / 1024**3:.1f} GB  "
          f"bf16={torch.cuda.is_bf16_supported()}")
    expected_name = gpu_selection.get('selected_name')
    if expected_name and expected_name != properties.name:
        raise RuntimeError(
            f"Selected GPU {physical} is '{expected_name}' in nvidia-smi but torch sees "
            f"'{properties.name}'. The device selection did not take effect; restart the "
            "kernel and Run All."
        )


In [ ]:
from puma.runtime import create_runtime, preflight_environment

runtime = create_runtime(
    PROJECT_DIR,
    run_folds=(0, 1, 2, 3, 4),
    seeds=(0,),
    stage1_epochs=40,
    stage2_epochs=50,
    stage1_effective_batch_size=16,
    stage2_effective_batch_size=256,
    stage1_micro_batch_size=16,
    stage2_micro_batch_size=256,
    preprocessing_workers=0,  # 0 = all logical CPU cores
)
print(runtime.as_dict())

preflight_report = preflight_environment(
    runtime,
    require_dataset=True,
    require_training_dependencies=False,
)


In [ ]:
from puma.data.preprocess import preprocess_dataset

FORCE_PREPROCESS = False
artifacts = preprocess_dataset(runtime, force=FORCE_PREPROCESS)
artifacts


In [ ]:
# Fold integrity. Stage 1 uses each fold twice: as the untouched outer fold that receives
# the OOF prediction, and as another fold's inner split for checkpoint / heatmap-threshold
# / local-max-radius selection. A lopsided split trains and reports without any error, so
# it is checked explicitly here before training starts.
import numpy as np
import pandas as pd
from puma.config import PUMA_CLASS_NAMES
from puma.data.datasets import PumaNpyStore
from puma.data.preprocess import validate_fold_assignments

store = PumaNpyStore.open(runtime.paths.artifact_dir)
folds = np.asarray(store.folds)
manifest = store.manifest

# Raises ValueError on a degenerate split. run_stage1_a1() runs the same check.
summary = validate_fold_assignments(folds, runtime.data.number_of_folds)
print('fold sizes           :', summary['fold_sizes'])
print('expected size / fold :', summary['expected_fold_size'])
print('size imbalance ratio :', summary['size_imbalance_ratio'], '(1.0 = perfect)')

rows = []
for fold in range(runtime.data.number_of_folds):
    mask = folds == fold
    counts = np.array([manifest[f'count_class_{i}'][mask].sum() for i in range(len(PUMA_CLASS_NAMES))])
    row = {
        'fold': fold,
        'rois': int(mask.sum()),
        'nuclei': int(counts.sum()),
        'primary': int((manifest['melanoma_type'][mask] == 'primary').sum()),
        'metastatic': int((manifest['melanoma_type'][mask] == 'metastatic').sum()),
    }
    row.update({name.replace('nuclei_', ''): int(value)
                for name, value in zip(PUMA_CLASS_NAMES, counts)})
    rows.append(row)
fold_table = pd.DataFrame(rows).set_index('fold')
display(fold_table)

empty = {
    fold: [name for name in (n.replace('nuclei_', '') for n in PUMA_CLASS_NAMES)
           if fold_table.loc[fold, name] == 0]
    for fold in fold_table.index
}
missing = {fold: names for fold, names in empty.items() if names}
print('folds missing a class entirely:', missing or 'none')
